# Creating a loader with Kosh

In [1]:
import os
import kosh

# Make sure local file is new sql file
kosh_example_sql_file = "kosh_ultra_example.sql"
    
# Create db on file
kosh.create_new_db(kosh_example_sql_file)

In [2]:
from  kosh import KoshStore, KoshLoader
import os

# connect to store
store = KoshStore(engine="sina", username=os.environ["USER"], sql='sql', db_path=kosh_example_sql_file)

['94a9ac3dd3c440eca0d105d14246ce4f']


In [3]:
# Add MNST datasets to store
sample = store.create(name="example", metadata={'project':"example"})


In [4]:
# Associate files with datasets
sample.add_file("example.ultra", mime_type="ultra")

In [5]:
import numpy

class UltraLoader(KoshLoader):
    def load_from_ultra(self, variable):
        if not isinstance(variable, (list,tuple)):  # only one variable requested
            variable = [variable,]
        variables = [[],] * len(variable)
        previous_line = ""
        var_names = None
        with open(self.obj.uri, "rb") as f:
            for line in f.readlines():
                # Skip headers
                line = line.decode("utf-8")
                if line[0]=="#" or line.strip()=="end":
                    previous_line = line
                    var_names = None
                    continue
                if var_names is None:
                    var_names = previous_line.split()[1:]
                    # clean up name list
                    while "vs" in var_names:
                        var_names.remove("vs")
                sp = line.split()
                for ivar, name in enumerate(variable):
                    if name in var_names:
                        index = var_names.index(name)
                        variables[ivar].append(float(sp[index]))
        # we're done reading these variables, co
        for ivar in range(len(variables)):
            if len(variables[ivar]) > 0 and isinstance(variable[ivar], list):
                variables[ivar] = numpy.array(variables[ivar])
        if len(variables) > 1:
            return variables
        else:  # only one variable read in
            return variables[0]
        
    def __init__(self, obj):
        super(UltraLoader, self).__init__(obj, {"ultra": ["numpy", ]})

    def get(self, variables, *args, **kargs):
        return self.load_from_ultra(variables)
        
    def list_features(self):
        variables = set()
        with open(self.obj.uri, "r") as f:
            previous = ""
            for line in f.readlines():
                if line[0] == "#":
                    previous = line
                    var_names = None
                    continue
                if var_names is not None:
                    continue
                var_names = previous.split()[1:]
                while "vs" in var_names:
                    var_names.remove("vs")
                for name in var_names:
                    variables.add(name)
        return list(variables)    

store.add_loader(UltraLoader)

In [6]:
print(sample.list_features())
print(sample.get("energy"))

['energy', 'time', 'var2']
[0.6, 0.7, 0.8, 0.6, 0.5, 0.2, 0.1, 0.6]
